In [1]:
# Import libraries
import pandas as pd
import numpy as np

In [2]:
# Import datasets
production = pd.read_parquet('../data/02_intermediate/production.parquet')
transactions = pd.read_parquet('../data/02_intermediate/transactions.parquet')
default_df = pd.read_parquet("../data/02_intermediate/default.parquet")

In [3]:
assert production.shape == (68644, 20)
assert transactions.shape[0] > 1_000_000
assert default_df.shape == (68644, 8)

# Phase 2 — Behavioral features

Compress up to 12 months of payment history into **157 customer-level columns** (rolling agr/ags windows + act* counters). One row per `cid` per application month.

See `documents/plan.md` Phase 2 for the modeling rationale.

In [4]:
# Phase 2 parameters — keep in sync with conf/base/parameters.yml (behavioral section)
PARAMS = {
    "max_length": 12,          # months of raw payment history in the pivot window
    "window_lengths": [3, 6, 9, 12],  # rolling summary windows (industry standard set)
    "stats": ["Mean", "Max", "Min"],  # aggregate stats applied inside each window
    "base_vars": [  # six customer-month risk streams (Ins / Css / All × Days / Due)
        "CMaxI_Days", "CMaxI_Due", "CMaxC_Days", "CMaxC_Due", "CMaxA_Days", "CMaxA_Due"
    ],
    "grace_days": 15,            # shift pay_days so on-time-by-15th → days=0
    "nmiss_threshold": 1,        # agr* blanked when >1 month missing in window
    "product_streams": [
        {"prefix": "CMaxI", "product": "ins"},
        {"prefix": "CMaxC", "product": "css"},
        {"prefix": "CMaxA", "product": None},  # None = all products combined
    ],
}


### `PARAMS` — what each knob controls

| Parameter | Value | Why it exists |
|-----------|-------|---------------|
| `max_length` | 12 | Months of raw history in the pivot — matches industry "last 12 months" convention |
| `window_lengths` | 3,6,9,12 | Multiple horizons — recent trouble (3m) vs structural pattern (12m); WOE picks winners |
| `stats` | Mean, Max, Min | Max = worst moment; Mean = typical behavior; Min = best month (recovery signal) |
| `base_vars` | 6 CMax* streams | Ins/Css/All × Days/Due — product-specific and combined risk |
| `grace_days` | 15 | Align lateness to "past the 15th" policy window (see `filter_transactions` note) |
| `nmiss_threshold` | 1 | `agr*` strict rule: >1 missing month in window → feature blanked → Missing bin |
| `product_streams` | I / C / A | Three parallel pivots merged on `cid` |

These land in `conf/base/parameters.yml` when ports to Kedro — one source of truth


In [5]:
# --- Period helpers ---

def derive_proc_period1(proc_period: str) -> str:
    """Convert application month → last month of usable behavioral history.

    Actions:
    1. Parse proc_period as a monthly Period (YYYYMM).
    2. Subtract one month — features must not include the application month itself
       (that would be look-ahead leakage).
    3. Return as YYYYMM string (e.g. 197502 → 197501).
    """
    return (pd.Period(proc_period, freq="M") - 1).strftime("%Y%m")


def derive_proc_periodf(proc_period1: str, max_length: int) -> str:
    """First snapshot month in the 12-month behavioral lookback window.

    Actions:
    1. Anchor on proc_period1 (end of history).
    2. Step back (max_length - 2) months — matches the SAS window that feeds
       agr12_* / ags12_* (12 months ending at proc_period1).
    3. Return YYYYMM string (e.g. 197501 → 197403).
    """
    anchor = pd.Period(proc_period1, freq="M")
    return (anchor - max_length + 2).strftime("%Y%m")


def get_cust_unique(production: pd.DataFrame, proc_period: str) -> pd.Series:
    """Customer IDs with at least one application in proc_period.

    Actions:
    1. Filter production to rows where period == proc_period.
    2. Return deduplicated cid values (one row per customer, not per loan).
    """
    mask = production["period"] == proc_period
    return production.loc[mask, "cid"].drop_duplicates().reset_index(drop=True)


# Quick smoke test
proc_period = "197502"
proc_period1 = derive_proc_period1(proc_period)
proc_periodf = derive_proc_periodf(proc_period1, PARAMS["max_length"])
cust_unique = get_cust_unique(production, proc_period)

print(f"Application month (proc_period):     {proc_period}")
print(f"Behavioral cutoff (proc_period1):    {proc_period1}")
print(f"History window start (proc_periodf): {proc_periodf}")
print(f"Applicants this month (unique cid):  {len(cust_unique)}")


Application month (proc_period):     197502
Behavioral cutoff (proc_period1):    197501
History window start (proc_periodf): 197403
Applicants this month (unique cid):  294


### Period helpers — why three different months?

Credit scoring simulation uses **three month labels**. Confusing them is the #1 source of bugs in this project.

| Symbol | Example (`proc_period=197502`) | Role |
|--------|-------------------------------|------|
| **`proc_period`** | `197502` | Application month — when the customer applied |
| **`proc_period1`** | `197501` | Last month of **usable** behavioral history |
| **`proc_periodf`** | `197403` | First month in the 12-month lookback window |

**Why `proc_period1 = proc_period − 1`?**

At application time in February 1975, the bank only knows payments through January 1975. Using February snapshots would mean the model sees how the customer behaved *after* they applied — **target leakage**. In an interview: *"Features must be point-in-time; behavioral cutoff is always month before application."*

**Why `proc_periodf`?**

We don't use the customer's entire life history — only the last 12 months of snapshots ending at `proc_period1`. Older behavior is less predictive and creates sparse columns. The rolling `agr12_*` features need exactly this window.

**`get_cust_unique`**

Returns **customers** who applied this month, not loans. One customer can submit multiple applications; behavioral features are built per `cid`, then joined to every `aid` in Phase 3.


In [6]:
def filter_transactions(transaction: pd.DataFrame,
                        cust_unique: pd.Series,
                        proc_periodf: str,
                        proc_period1: str,
                        grace_days: int = 15) -> pd.DataFrame:
    """Slice transactions to the behavioral window and derive the days indicator.

    Actions:
    1. Keep rows whose cid is in cust_unique (only applicants this month).
    2. Keep rows whose snapshot period is in [proc_periodf, proc_period1].
    3. Derive days = pay_days + grace_days (SAS: pay_days+15; no lower clip).
       Negative values mean paid before the grace window — kept intentionally.
    4. Return a copy; due_installments stays as the due counter.
    """
    mask_cid = transaction["cid"].isin(cust_unique)
    mask_period = (transaction["period"] >= proc_periodf) & (transaction["period"] <= proc_period1)
    filtered_df = transaction.loc[mask_cid & mask_period].copy()

    filtered_df["days"] = filtered_df["pay_days"] + grace_days

    return filtered_df


transaction_filtered = filter_transactions(transactions, cust_unique, proc_periodf, proc_period1)

print(f"Filtered transaction rows:  {len(transaction_filtered):,}")
print(f"Unique customers covered:   {transaction_filtered['cid'].nunique():,}")
print(f"Period range:               {transaction_filtered['period'].min()} → {transaction_filtered['period'].max()}")
print(f"days range:                 {transaction_filtered['days'].min()} → {transaction_filtered['days'].max()}")


Filtered transaction rows:  6,553
Unique customers covered:   184
Period range:               197403 → 197501
days range:                 3.0 → 25.0


### `filter_transactions` — what the checkpoint numbers mean

**Period range `[proc_periodf, proc_period1]`**

For applications in `197502`, we only use payment snapshots through `197501`. That guarantees **no look-ahead leakage** — we never use payment behavior from the application month or after.

**`days = pay_days + 15` (no lower clip)**

| `pay_days` | `days` | Interpretation |
|------------|--------|----------------|
| −10 | 5 | Paid 10 days early → still inside grace window |
| 0 | 15 | Paid exactly on due date → at grace boundary |
| +10 | 25 | 10 days late past due date → 10 days past grace |
| NaN | NaN | No payment recorded that month → missing history |

We keep negative `pay_days` results (small positive `days`) instead of clipping to 0, matching the SAS reference (`pay_days+15`). Clipping would erase the "paid early" vs "paid on time" distinction inside grace.

**Row count vs unique customers:** multiple rows per customer is expected — one row per **loan-month**. The pivot step collapses to one value per customer per month via `max`.


In [11]:
def pivot_one_stream(transaction_filtered: pd.DataFrame, stream: dict) -> pd.DataFrame:
    """Pivot one product stream to wide customer-month columns.

    Actions:
    1. Filter to product stream (ins, css, or all if product is None).
    2. Aggregate to (cid, period) taking max(days) and max(due_installments)
       — worst loan dominates, per credit-policy convention.
    3. Pivot wide: one row per cid, columns {prefix}_Days_{YYYYMM} / {prefix}_Due_{YYYYMM}.
    4. Missing months stay NaN (filled later by rolling logic, not with 0).
    """
    prefix = stream["prefix"]
    product = stream["product"]

    if product is None:
        stream_df = transaction_filtered
    else:
        stream_df = transaction_filtered[transaction_filtered["product"] == product]

    grouped = (
        stream_df.groupby(["cid", "period"])
        .agg(days=("days", "max"), due_installments=("due_installments", "max"))
        .reset_index()
    )

    pivot = grouped.pivot(index="cid", columns="period", values=["days", "due_installments"])
    pivot.columns = [
        f"{prefix}_{'Days' if metric == 'days' else 'Due'}_{period}"
        for metric, period in pivot.columns
    ]

    return pivot.reset_index()


def pivot_customer_monthly(transaction_filtered: pd.DataFrame,
                            cust_unique: pd.Series,
                            product_streams: list) -> pd.DataFrame:
    """Build the wide abt_beh scaffold: cid + up to 72 monthly columns.

    Actions:
    1. Start with one row per applicant cid (left spine from cust_unique).
    2. For each product stream (Ins / Css / All), pivot independently.
    3. Left-merge each stream on cid — customers with no css history get NaN css cols.
    """
    abt_beh = pd.DataFrame({"cid": cust_unique.values})

    for stream in product_streams:
        stream_pivot = pivot_one_stream(transaction_filtered, stream)
        abt_beh = abt_beh.merge(stream_pivot, on="cid", how="left")

    return abt_beh


def build_period_list(proc_period1: str, window: int) -> list:
    """Return chronological list of YYYYMM months in a rolling window.

    Actions:
    1. Anchor on proc_period1 (last month in window).
    2. Walk back (window - 1) months.
    3. Return oldest-first (e.g. build_period_list('197501', 3) → ['197411','197412','197501']).
    """
    anchor = pd.Period(proc_period1, freq="M")
    periods = [(anchor - i).strftime("%Y%m") for i in range(window)]
    return list(reversed(periods))


def make_rolling_features(abt_beh: pd.DataFrame,
                            proc_period1: str,
                            params: dict) -> pd.DataFrame:
    """Compress monthly columns into 144 rolling-window features (72 ags + 72 agr).

    Actions (for each base_var × window × stat):
    1. Collect source columns {base_var}_{YYYYMM} for the window ending at proc_period1.
    2. Count nmiss = NaN months in window + months with no column at all.
    3. Compute ags* (lenient): Mean/Max/Min with skipna=True over available months.
    4. Compute agr* (strict): copy ags*, but set to NaN when nmiss > nmiss_threshold
       — signals "not enough history" to the WOE binner's Missing bucket.
    """
    window_lengths = params["window_lengths"]
    stats = params["stats"]
    base_vars = params["base_vars"]
    nmiss_threshold = params["nmiss_threshold"]

    columns = {"cid": abt_beh["cid"]}

    for base_var in base_vars:
        for window in window_lengths:
            window_periods = build_period_list(proc_period1, window)
            window_cols = [f"{base_var}_{period}" for period in window_periods]

            existing_cols = [c for c in window_cols if c in abt_beh.columns]
            missing_cols = [c for c in window_cols if c not in abt_beh.columns]

            if existing_cols:
                window_data = abt_beh[existing_cols]
            else:
                window_data = pd.DataFrame(np.nan, index=abt_beh.index, columns=window_cols)

            nmiss = window_data.isna().sum(axis=1) + len(missing_cols)

            for stat in stats:
                ags_name = f"ags{window}_{stat}_{base_var}"
                agr_name = f"agr{window}_{stat}_{base_var}"

                if stat == "Mean":
                    ags_values = window_data.mean(axis=1, skipna=True)
                elif stat == "Max":
                    ags_values = window_data.max(axis=1, skipna=True)
                elif stat == "Min":
                    ags_values = window_data.min(axis=1, skipna=True)
                else:
                    raise ValueError(f"Unknown stat: {stat}")

                agr_values = ags_values.where(nmiss <= nmiss_threshold, np.nan)

                columns[ags_name] = ags_values
                columns[agr_name] = agr_values

    return pd.DataFrame(columns)


In [12]:
cust_unique = get_cust_unique(production, proc_period)
transaction_filtered = filter_transactions(transactions, cust_unique, proc_periodf, proc_period1)

abt_beh = pivot_customer_monthly(transaction_filtered, cust_unique, PARAMS["product_streams"])
abt_rolling = make_rolling_features(abt_beh, proc_period1, PARAMS)

display(cust_unique.head(3))
display(transaction_filtered.head(3))
display(abt_beh.head(3))
display(abt_rolling.head(3))

0    0000000713
1    0000001381
2    0000002567
Name: cid, dtype: str

,cid,aid,product,period,fin_period,status,due_installments,paid_installments,pay_days,income,n_installments,spendings,installment,leftn_installments,days
44097,0000001297,ins1971010400011,ins,197403,197101,C,0.0,36.0,-1.0,1933.0,36.0,400.0,210.0,0.0,14.0
50239,0000001479,ins1971022400031,ins,197403,197102,A,2.0,35.0,NaN,859.0,36.0,240.0,140.0,1.0,NaN
50240,0000001479,ins1971022400031,ins,197404,197102,A,3.0,35.0,NaN,859.0,36.0,240.0,140.0,1.0,NaN


,cid,CMaxI_Days_197403,CMaxI_Days_197404,CMaxI_Days_197405,CMaxI_Days_197406,CMaxI_Days_197407,CMaxI_Days_197408,CMaxI_Days_197409,CMaxI_Days_197410,CMaxI_Days_197411,...,CMaxA_Due_197404,CMaxA_Due_197405,CMaxA_Due_197406,CMaxA_Due_197407,CMaxA_Due_197408,CMaxA_Due_197409,CMaxA_Due_197410,CMaxA_Due_197411,CMaxA_Due_197412,CMaxA_Due_197501
0,0000000713,15.0,8.0,18.0,11.0,15.0,14.0,NaN,9.0,12.0,...,2.0,2.0,3.0,4.0,5.0,6.0,7.0,7.0,1.0,0.0
1,0000001381,12.0,15.0,13.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,2.0
2,0000002567,9.0,8.0,9.0,13.0,15.0,14.0,12.0,14.0,9.0,...,1.0,2.0,1.0,1.0,2.0,2.0,2.0,2.0,2.0,3.0


,cid,ags3_Mean_CMaxI_Days,agr3_Mean_CMaxI_Days,ags3_Max_CMaxI_Days,agr3_Max_CMaxI_Days,ags3_Min_CMaxI_Days,agr3_Min_CMaxI_Days,ags6_Mean_CMaxI_Days,agr6_Mean_CMaxI_Days,ags6_Max_CMaxI_Days,...,ags9_Max_CMaxA_Due,agr9_Max_CMaxA_Due,ags9_Min_CMaxA_Due,agr9_Min_CMaxA_Due,ags12_Mean_CMaxA_Due,agr12_Mean_CMaxA_Due,ags12_Max_CMaxA_Due,agr12_Max_CMaxA_Due,ags12_Min_CMaxA_Due,agr12_Min_CMaxA_Due
0,0000000713,14.0,14.0,15.0,15.0,12.0,12.0,13.0,13.0,15.0,...,7.0,7.0,0.0,0.0,3.454545,3.454545,7.0,7.0,0.0,0.0
1,0000001381,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2.0,2.0,0.0,0.0,0.727273,0.727273,2.0,2.0,0.0,0.0
2,0000002567,10.5,10.5,12.0,12.0,9.0,9.0,12.2,12.2,14.0,...,3.0,3.0,1.0,1.0,1.727273,1.727273,3.0,3.0,1.0,1.0


### `pivot_one_stream` — why max, and why Days + Due?

**Why aggregate with `max` per (cid, period)?**

A customer can hold multiple loans in the same month. Risk is driven by the **worst** loan, not the average:

- Customer with two perfect ins loans and one css loan 60 days late → `max(days)` catches the css trouble.
- `mean` would dilute that signal — a common beginner mistake.

**Why two variables per stream?**

| Variable | What it captures | Example signal |
|----------|------------------|----------------|
| **Days** (`pay_days + 15`) | *How late* payments are — severity / timing | Paid 30 days late vs paid on the 14th inside grace |
| **Due** (`due_installments`) | *How many* installments missed — counter / persistence | `due=0` good, `due≥1` in arrears, `due>3` → `default12=1` label |

Together they encode **frequency** (Due) and **severity** (Days) — the two dimensions credit analysts ask about in every collections review.

**Why three streams (Ins / Css / All)?**

- Ins and css have different repayment dynamics (12–36 mo installment vs shorter cash loan).
- A clean css delinquency signal should not be averaged away by perfect ins behavior.
- `CMaxA_*` (All) powers the `act*_n_arrears*` counters — any-product trouble.

**Wide format:** each month becomes a column (`CMaxI_Days_197501`) so we can roll 3/6/9/12-month windows backward from `proc_period1`. Those rolls become the `agr*` / `ags*` inputs to the scorecard.


In [13]:
def add_counter_features(abt_beh: pd.DataFrame,
                        proc_period1: str,
                        params: dict) -> pd.DataFrame:
    """Hand-crafted policy counters on the All-product (CMaxA) stream.

    Actions (for each window in {3,6,9,12}):
    1. Pull CMaxA_Due_{p} and CMaxA_Days_{p} for months in the window.
    2. act{n}_n_arrears      = count months where due >= 1 (any delinquency).
    3. act{n}_n_arrears_days  = count months where days > 15 (past grace period).
    4. act{n}_n_good_days     = count months where 0 < days < 15 (paid inside grace).
    These three consistently rank among top-IV features in the reference scorecards.
    """
    columns = {"cid": abt_beh["cid"]}

    for window in params["window_lengths"]:
        window_periods = build_period_list(proc_period1, window)

        due_cols = [f"CMaxA_Due_{p}" for p in window_periods]
        days_cols = [f"CMaxA_Days_{p}" for p in window_periods]

        due_existing = [c for c in due_cols if c in abt_beh.columns]
        days_existing = [c for c in days_cols if c in abt_beh.columns]

        due_data = abt_beh[due_existing] if due_existing else pd.DataFrame(np.nan, index=abt_beh.index, columns=due_cols)
        days_data = abt_beh[days_existing] if days_existing else pd.DataFrame(np.nan, index=abt_beh.index, columns=days_cols)

        columns[f"act{window}_n_arrears"] = (due_data >= 1).sum(axis=1)
        columns[f"act{window}_n_arrears_days"] = (days_data > 15).sum(axis=1)
        columns[f"act{window}_n_good_days"] = ((days_data > 0) & (days_data < 15)).sum(axis=1)

    return pd.DataFrame(columns)


def make_abt(abt_beh: pd.DataFrame,
              proc_period1: str,
              params: dict) -> pd.DataFrame:
    """Combine rolling + counter features into the 157-column behavioral table.

    Actions:
    1. Call make_rolling_features → 145 cols (cid + 72 ags + 72 agr).
    2. Call add_counter_features → 13 cols (cid + 12 act* counters).
    3. Merge on cid (one-to-one). Raw CMax*_YYYYMM pivot columns are dropped.
    4. Assert shape: 1 + 72 + 72 + 12 = 157 columns.
    """
    abt_rolling = make_rolling_features(abt_beh, proc_period1, params)
    abt_counters = add_counter_features(abt_beh, proc_period1, params)

    abt = abt_rolling.merge(abt_counters, on="cid", how="left", validate="one_to_one")

    expected_cols = 1 + 72 + 72 + 12
    assert abt.shape[1] == expected_cols, f"Expected {expected_cols} columns, got {abt.shape[1]}"
    assert abt["cid"].is_unique, "cid is not unique in final ABT"

    return abt


def build_behavioral_for_month(transaction: pd.DataFrame,
                                 production: pd.DataFrame,
                                 proc_period: str,
                                 proc_period1: str,
                                 params: dict) -> pd.DataFrame:
    """End-to-end behavioral feature build for one application month.

    Actions:
    1. Derive proc_periodf (start of 12-month lookback).
    2. Identify applicants (cust_unique) from production.
    3. Filter transaction rows to [proc_periodf, proc_period1] for those cids.
       In Phase 3, transaction is the approved_tx pool (not raw transactions).
    4. Pivot → roll → counter → return 157-column table keyed by cid.
    """
    proc_periodf = derive_proc_periodf(proc_period1, params["max_length"])
    cust_unique = get_cust_unique(production, proc_period)

    transaction_filtered = filter_transactions(
        transaction, cust_unique, proc_periodf, proc_period1, params["grace_days"]
    )

    abt_beh = pivot_customer_monthly(transaction_filtered, cust_unique, params["product_streams"])
    return make_abt(abt_beh, proc_period1, params)


In [14]:
abt_beh = pivot_customer_monthly(transaction_filtered, cust_unique, PARAMS["product_streams"])
abt = make_abt(abt_beh, proc_period1, PARAMS)

display(abt.head(3))

,cid,ags3_Mean_CMaxI_Days,agr3_Mean_CMaxI_Days,ags3_Max_CMaxI_Days,agr3_Max_CMaxI_Days,ags3_Min_CMaxI_Days,agr3_Min_CMaxI_Days,ags6_Mean_CMaxI_Days,agr6_Mean_CMaxI_Days,ags6_Max_CMaxI_Days,...,act3_n_good_days,act6_n_arrears,act6_n_arrears_days,act6_n_good_days,act9_n_arrears,act9_n_arrears_days,act9_n_good_days,act12_n_arrears,act12_n_arrears_days,act12_n_good_days
0,0000000713,14.0,14.0,15.0,15.0,12.0,12.0,13.0,13.0,15.0,...,1,5,0,4,8,1,4,10,1,5
1,0000001381,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1,4,0,3,7,0,6,7,0,6
2,0000002567,10.5,10.5,12.0,12.0,9.0,9.0,12.2,12.2,14.0,...,1,6,2,2,9,2,3,11,2,3


### `make_abt` — from 72 monthly columns to 157 model features

**Rolling features (144 cols: 72 `ags*` + 72 `agr*`):**

| Prefix | Meaning | Modeling use |
|--------|---------|--------------|
| `ags3_Mean_CMaxI_Days` | Lenient 3-month average lateness on ins loans | Uses whatever months exist — good for sparse history |
| `agr3_Mean_CMaxI_Days` | Strict same stat | **Blanked** when >1 month missing → explicit "not enough history" for WOE Missing bin |
| `agr12_Max_CMaxA_Due` | Worst missed-installment count over 12 months | Captures "ever been seriously delinquent recently" |

**Why `agr` AND `ags`?** Let the WOE binner decide which missingness rule is more predictive per variable — cheaper than hand-picking.

**Counter features (12 cols):**

| Column | Policy question it encodes |
|--------|---------------------------|
| `act12_n_arrears` | How many months had *any* delinquency? (frequency of trouble) |
| `act12_n_arrears_days` | How many months past the 15-day grace window? (moderate severity) |
| `act12_n_good_days` | Paid inside grace but not early — "borderline" payers |

These three consistently rank **top IV** in the reference scorecards — they bridge raw payment data to credit-policy language regulators understand.

**157-column assert** = Phase 2 complete for one month. Next: wire these into the monthly simulation loop.


In [15]:
def build_behavioral_all_months(transaction: pd.DataFrame,
                                  production: pd.DataFrame,
                                  params: dict,
                                  output_dir: str = ".",
                                  start_period: str = "197502",
                                  end_period: str = "198712") -> list:
    """Optional batch export: one behavioral parquet per application month.

    Actions:
    1. Loop proc_period from start_period to end_period.
    2. For each month, call build_behavioral_for_month and write
       behavioral_{proc_period1}.parquet to output_dir.
    3. Return summary list of (proc_period, proc_period1, path, n_rows).

    Note: uses the transaction argument as-is. For exploration this is raw
    transactions; in the Phase 3 simulation loop the same function receives
    approved_tx instead (reject-inference-aware history).
    """
    all_periods = sorted(production["period"].unique())
    target_periods = [p for p in all_periods if start_period <= p <= end_period]

    results = []
    for proc_period in target_periods:
        proc_period1 = derive_proc_period1(proc_period)
        abt = build_behavioral_for_month(
            transaction, production, proc_period, proc_period1, params
        )
        out_path = f"{output_dir}/behavioral_{proc_period1}.parquet"
        abt.to_parquet(out_path, index=False)
        results.append((proc_period, proc_period1, out_path, len(abt)))
        print(f"{proc_period} -> {proc_period1}: {len(abt):>6} rows -> {out_path}")

    return results


In [16]:
results = build_behavioral_all_months(
    transactions, production, PARAMS,
    output_dir="../data/03_primary",
    start_period="197502",
    end_period="198712",
)

# Checkpoint
summary = pd.DataFrame(results, columns=["proc_period", "proc_period1", "path", "n_rows"])
display(summary)
assert summary["proc_period1"].is_unique
assert (summary["n_rows"] > 0).all()

197502 -> 197501:    294 rows -> ../data/03_primary/behavioral_197501.parquet
197503 -> 197502:    313 rows -> ../data/03_primary/behavioral_197502.parquet
197504 -> 197503:    302 rows -> ../data/03_primary/behavioral_197503.parquet
197505 -> 197504:    314 rows -> ../data/03_primary/behavioral_197504.parquet
197506 -> 197505:    312 rows -> ../data/03_primary/behavioral_197505.parquet
197507 -> 197506:    310 rows -> ../data/03_primary/behavioral_197506.parquet
197508 -> 197507:    312 rows -> ../data/03_primary/behavioral_197507.parquet
197509 -> 197508:    299 rows -> ../data/03_primary/behavioral_197508.parquet
197510 -> 197509:    316 rows -> ../data/03_primary/behavioral_197509.parquet
197511 -> 197510:    308 rows -> ../data/03_primary/behavioral_197510.parquet
197512 -> 197511:    374 rows -> ../data/03_primary/behavioral_197511.parquet
197601 -> 197512:    317 rows -> ../data/03_primary/behavioral_197512.parquet
197602 -> 197601:    307 rows -> ../data/03_primary/behavioral_1

,proc_period,proc_period1,path,n_rows
0,197502,197501,../data/03_primary/behavioral_197501.parquet,294
1,197503,197502,../data/03_primary/behavioral_197502.parquet,313
2,197504,197503,../data/03_primary/behavioral_197503.parquet,302
3,197505,197504,../data/03_primary/behavioral_197504.parquet,314
4,197506,197505,../data/03_primary/behavioral_197505.parquet,312
...,...,...,...,...
150,198708,198707,../data/03_primary/behavioral_198707.parquet,309
151,198709,198708,../data/03_primary/behavioral_198708.parquet,320
152,198710,198709,../data/03_primary/behavioral_198709.parquet,310
153,198711,198710,../data/03_primary/behavioral_198710.parquet,296


### `build_behavioral_all_months` — what did we just export?

Each parquet file `behavioral_{proc_period1}.parquet` is a **customer-level feature table** for one application month:

- **Rows** ≈ number of applicants that month (unique `cid`), not loans.
- **157 columns** = rolling windows (agr/ags) + policy counters (act*).

**Important caveat:** this batch job uses **raw `transactions`**, not the simulation's `approved_tx` pool. That's fine for exploring feature distributions, but **Phase 3 simulation must rebuild behavioral from `approved_tx`** — otherwise you ignore reject inference (rejected customers' payment history would still be visible).

In Kedro you can skip this export entirely and compute behavioral only inside `run_simulation`.


In [17]:
check = pd.read_parquet('../data/03_primary/behavioral_198711.parquet')
display(check.head())

,cid,ags3_Mean_CMaxI_Days,agr3_Mean_CMaxI_Days,ags3_Max_CMaxI_Days,agr3_Max_CMaxI_Days,ags3_Min_CMaxI_Days,agr3_Min_CMaxI_Days,ags6_Mean_CMaxI_Days,agr6_Mean_CMaxI_Days,ags6_Max_CMaxI_Days,...,act3_n_good_days,act6_n_arrears,act6_n_arrears_days,act6_n_good_days,act9_n_arrears,act9_n_arrears_days,act9_n_good_days,act12_n_arrears,act12_n_arrears_days,act12_n_good_days
0,0000001054,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1,6,1,3,8,2,4,10,2,5
1,0000001817,12.666667,12.666667,13.0,13.0,12.0,12.0,13.333333,13.333333,15.0,...,2,0,0,2,0,0,3,0,0,5
2,0000002643,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3,2,0,4,2,0,5,4,0,6
3,0000002805,14.000000,14.000000,15.0,15.0,12.0,12.0,14.000000,14.000000,15.0,...,1,0,0,3,0,0,4,0,0,4
4,0000000163,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3,6,0,5,9,0,7,11,0,9


# Phase 3 — ABT assembly & simulation

Month-by-month loop: behavioral features from the **approved pool** → customer aggregates → join to applications → benchmark decision engine → grow pool. Output: `abt_app` (one row per application).

See `documents/plan.md` Phase 3 and `documents/kedro_phase2_3_port.md` for the Kedro port plan.

In [18]:
# Phase 3 simulation parameters — keep in sync with conf/base/parameters.yml (simulation section)
SIM_PARAMS = {
    "start_period": "197501",   # first application month written to abt_app
    "end_period": "198712",     # last application month in modeling window
    "seed_period": None,        # None → auto: earliest fin_period in transactions (burn-in pool)
    "burn_in_before": "197501", # before this month, approve everyone (no css inactive rule)
}


In [19]:
def build_cust_product_agg(approved_tx: pd.DataFrame,
                            cust_unique: pd.Series,
                            proc_period1: str,
                            version: str) -> pd.DataFrame:
    """Per-customer product aggregates for ins or css (act_cins_* / act_ccss_*).

    Actions:
    Part A — Snapshot at period == proc_period1 (current portfolio state):
      1. Filter approved_tx to product and applicant cids.
      2. Keep rows at proc_period1 only.
      3. Group by cid: count active loans, max due, min paid/left installments.
      4. Compute utilization ratios across the customer's loans:
         utl = sum(paid)/sum(n_installments), dueutl = sum(due)/sum(n_installments),
         cc = (sum(installment)+sum(spendings))/sum(income).

    Part B — Lifetime history with period <= proc_period1:
      5. Compute seniority = months from fin_period to proc_period1 (+1).
      6. Aggregate max/min seniority, distinct aid count, count status B/C closures.

    Customers with no loans of this product are absent (NaN after left-join in assembly).
    """
    v = version
    base = approved_tx[
        (approved_tx["product"] == version) & (approved_tx["cid"].isin(cust_unique))
    ]

    snap = base[base["period"] == proc_period1]

    snap_agg = snap.groupby("cid").agg(
        **{
            f"act_c{v}_n_loans_act": ("aid", "count"),
            f"act_c{v}_maxdue": ("due_installments", "max"),
            f"act_c{v}_min_pninst": ("paid_installments", "min"),
            f"act_c{v}_min_lninst": ("leftn_installments", "min"),
        }
    )

    snap_sums = snap.groupby("cid").agg(
        paid_sum=("paid_installments", "sum"),
        due_sum=("due_installments", "sum"),
        n_sum=("n_installments", "sum"),
        installment_sum=("installment", "sum"),
        spendings_sum=("spendings", "sum"),
        income_sum=("income", "sum"),
    )

    snap_agg[f"act_c{v}_utl"] = snap_sums["paid_sum"] / snap_sums["n_sum"]
    snap_agg[f"act_c{v}_dueutl"] = snap_sums["due_sum"] / snap_sums["n_sum"]
    snap_agg[f"act_c{v}_cc"] = (
        snap_sums["installment_sum"] + snap_sums["spendings_sum"]
    ) / snap_sums["income_sum"]

    hist = base[base["period"] <= proc_period1].copy()
    anchor = pd.Period(proc_period1, freq="M")
    hist["seniority"] = hist["fin_period"].apply(
        lambda fp: (anchor - pd.Period(fp, freq="M")).n + 1
    )

    hist_agg = hist.groupby("cid").agg(
        **{
            f"act_c{v}_seniority": ("seniority", "max"),
            f"act_c{v}_min_seniority": ("seniority", "min"),
            f"act_c{v}_n_loans_hist": ("aid", "nunique"),
        }
    )

    n_statC = (
        hist[hist["status"] == "C"]
        .groupby("cid")["aid"].nunique()
        .rename(f"act_c{v}_n_statC")
    )
    n_statB = (
        hist[hist["status"] == "B"]
        .groupby("cid")["aid"].nunique()
        .rename(f"act_c{v}_n_statB")
    )

    out = (
        snap_agg
        .join(hist_agg, how="outer")
        .join(n_statC, how="outer")
        .join(n_statB, how="outer")
        .reset_index()
    )
    return out


def build_cust_active_flag(approved_tx: pd.DataFrame,
                            proc_period1: str) -> pd.DataFrame:
    """Flag customers with at least one active loan at proc_period1.

    Actions:
    1. Filter approved_tx to period == proc_period1 and status == 'A'.
    2. Deduplicate cid → act_cus_active = 1.
    3. Customers not in this table get NaN after left-join (= not active).

    Used by decision engine: css applicants with act_cus_active != 1 are declined.
    """
    snap = approved_tx[
        (approved_tx["period"] == proc_period1) & (approved_tx["status"] == "A")
    ]
    cids = snap["cid"].drop_duplicates()
    return pd.DataFrame({"cid": cids.values, "act_cus_active": 1}).reset_index(drop=True)


In [20]:
# Checkpoint 3a — product aggregates + active flag (uses full transactions as pool stand-in)
proc_period_test = "197502"
proc_period1_test = derive_proc_period1(proc_period_test)
cust_unique_test = get_cust_unique(production, proc_period_test)
approved_tx_standin = transactions  # real simulation uses seeded approved_tx pool

agg_ins = build_cust_product_agg(approved_tx_standin, cust_unique_test, proc_period1_test, "ins")
agg_css = build_cust_product_agg(approved_tx_standin, cust_unique_test, proc_period1_test, "css")
active_flag = build_cust_active_flag(approved_tx_standin, proc_period1_test)

assert "act_cins_utl" in agg_ins.columns
assert agg_ins["cid"].is_unique
assert agg_css["cid"].is_unique
assert active_flag["cid"].is_unique
assert active_flag["act_cus_active"].eq(1).all()

print(f"agg_ins:  {agg_ins.shape}")
print(f"agg_css:  {agg_css.shape}")
print(f"active:   {active_flag.shape}")


agg_ins:  (199, 13)
agg_css:  (161, 13)
active:   (3104, 2)


### `build_cust_product_agg` & `build_cust_active_flag` — what do these columns mean?

**Snapshot columns (`act_cins_*` / `act_ccss_*`)** — portfolio state at `proc_period1` (one month *before* the application):

| Column | Risk intuition |
|--------|----------------|
| `act_c{v}_utl` | How much of scheduled installments are already due — high = stretched |
| `act_c{v}_dueutl` | Missed-installment burden relative to contract length |
| `act_c{v}_cc` | Credit capacity on existing product loans — same idea as `act_cc` on the application form |
| `act_c{v}_maxdue` | Worst delinquency counter on any active loan right now |

**History columns** — lifetime behavior up to `proc_period1`:

| Column | Risk intuition |
|--------|----------------|
| `act_c{v}_seniority` | How long they've been a borrower — very new vs established |
| `act_c{v}_n_statB/C` | How many loans closed normally (B) vs charged off (C) — closure pattern |

**`act_cus_active`** — binary flag: did this customer have *any* loan with `status='A'` last month? Css cross-sell only makes sense for active customers; inactive css applicants get declined in v1.

Customers with no ins/css loans are **absent** from the agg table → NaN after left-join → WOE "Missing" bin in Phase 4 (first-time product takers).


In [21]:
def build_cust_all_agg(approved_tx: pd.DataFrame,
                        month_prod: pd.DataFrame,
                        cust_unique: pd.Series,
                        proc_period: str) -> pd.DataFrame:
    """Cross-product cumulative aggregates — one row per new application (aid).

    Actions:
    1. Collect currently active loans: approved_tx, status='A', period=proc_period.
    2. Append this month's new applications from month_prod (rename app_* → contract cols).
    3. Sort each customer's rows by origination time (aid[3:11] as YYYYMMDD proxy).
    4. Within cid, compute running sums of installment/spendings and loan counts by product.
    5. Derive act_call_cc = (installment_cum + spendings_cum) / income on each row.
    6. Keep only the new-application rows — merge key is aid, not cid.
    """
    active = approved_tx[
        (approved_tx["status"] == "A")
        & (approved_tx["period"] == proc_period)
        & (approved_tx["cid"].isin(cust_unique))
    ][["cid", "aid", "product", "installment", "spendings", "income"]].copy()
    active["is_new"] = False

    new_apps = month_prod[month_prod["cid"].isin(cust_unique)][
        ["cid", "aid", "product", "app_n_installments", "app_installment", "app_spendings", "app_income"]
    ].rename(columns={
        "app_n_installments": "n_installments",
        "app_installment": "installment",
        "app_spendings": "spendings",
        "app_income": "income",
    }).copy()
    new_apps["is_new"] = True

    combined = pd.concat([active, new_apps], ignore_index=True)
    combined["time"] = combined["aid"].str[3:11]
    combined = combined.sort_values(["cid", "time"]).reset_index(drop=True)

    combined["is_ins"] = (combined["product"] == "ins").astype(int)
    combined["is_css"] = (combined["product"] == "css").astype(int)

    grp = combined.groupby("cid")
    combined["installment_cum"] = grp["installment"].cumsum()
    combined["spendings_cum"] = grp["spendings"].cumsum()
    combined["act_cins_n_loan"] = grp["is_ins"].cumsum()
    combined["act_ccss_n_loan"] = grp["is_css"].cumsum()
    combined["act_call_n_loan"] = combined["act_cins_n_loan"] + combined["act_ccss_n_loan"]
    combined["act_call_cc"] = (
        combined["installment_cum"] + combined["spendings_cum"]
    ) / combined["income"]

    return combined.loc[
        combined["is_new"],
        ["aid", "act_call_cc", "act_cins_n_loan", "act_ccss_n_loan", "act_call_n_loan"],
    ].reset_index(drop=True)


def assemble_abt_month(month_prod: pd.DataFrame,
                       behavioral: pd.DataFrame,
                       agg_all: pd.DataFrame,
                       agg_ins: pd.DataFrame,
                       agg_css: pd.DataFrame,
                       active_flag: pd.DataFrame,
                       proc_period: str) -> pd.DataFrame:
    """Join application + behavioral + customer aggregates into one modeling row per aid.

    Actions:
    1. Start from month_prod (every application this month).
    2. Left-merge agg_all on aid (loan-level cumulative debt ratios).
    3. Left-merge behavioral on cid (157 customer-level payment-history features).
    4. Left-merge agg_ins / agg_css / active_flag on cid (portfolio snapshots).
    5. Stamp period = proc_period. Assert aid uniqueness.
    """
    abt = (
        month_prod
        .merge(agg_all, on="aid", how="left")
        .merge(behavioral, on="cid", how="left")
        .merge(agg_ins, on="cid", how="left")
        .merge(agg_css, on="cid", how="left")
        .merge(active_flag, on="cid", how="left")
    )
    abt["period"] = proc_period
    assert abt["aid"].is_unique, "duplicate aid after assembly — check merge keys"
    return abt


In [22]:
# Checkpoint 3c — assemble one month with real behavioral features
month_prod = production[production["period"] == proc_period_test]
behavioral_test = build_behavioral_for_month(
    approved_tx_standin, production, proc_period_test, proc_period1_test, PARAMS
)
agg_all = build_cust_all_agg(approved_tx_standin, month_prod, cust_unique_test, proc_period_test)
abt_month = assemble_abt_month(
    month_prod, behavioral_test, agg_all, agg_ins, agg_css, active_flag, proc_period_test
)

assert abt_month["aid"].is_unique
assert abt_month.shape[1] > 150
print(f"abt_month shape: {abt_month.shape}")
abt_month[["aid", "cid", "product", "agr3_Mean_CMaxI_Days", "act_cins_utl", "act_cus_active"]].head()


abt_month shape: (301, 205)


,aid,cid,product,agr3_Mean_CMaxI_Days,act_cins_utl,act_cus_active
0,css1975020100063,0000000713,css,14.000000,0.958333,1.0
1,css1975020100098,0000001381,css,NaN,NaN,1.0
2,css1975020100120,0000002567,css,10.500000,NaN,1.0
3,css1975020100123,0000002718,css,12.666667,0.972222,1.0
4,css1975020200009,0000000089,css,NaN,NaN,1.0


### `build_cust_all_agg` — cumulative debt at application time

This is the **hardest aggregate** in Phase 3 because the grain is `aid` (each new loan), not `cid`.

**What it computes:** for each new application, walk through the customer's loans in origination order and build **running totals**:

```
act_call_cc = (cumulative installments + cumulative spendings) / income on this row
act_cins_n_loan / act_ccss_n_loan / act_call_n_loan = how many ins/css/total loans so far
```

**Why include the new application in the cumulative sum?**

The bank decides *before* disbursement but *knows* the contract terms (`app_installment`, `app_income`). `act_call_cc` answers: *"If we approve this loan, what will their total debt burden be?"* — the same question an underwriter asks.

**Why sort by `aid[3:11]`?** Loan IDs embed origination date (`css19740205…` → 1974-02-05). Cumulative order must match real origination sequence.

**EDA Q10 link:** customers holding ins + css simultaneously get separate cumulative counts per product — risk is customer-level, not product-siloed.


### `assemble_abt_month` — why these joins?

The ABT grain is **one row per application (`aid`)**, but features live at different grains:

| Source | Grain | Join key | What it adds |
|--------|-------|----------|--------------|
| `month_prod` | aid | — | Application form (`app_*`, `act_*` at apply time) |
| `agg_all` | aid | `aid` | **This loan's** cumulative debt context (includes prior active loans + this application) |
| `behavioral` | cid | `cid` | **Customer's** 12-month payment *pattern* (agr*, ags*, act*) |
| `agg_ins` / `agg_css` | cid | `cid` | **Portfolio snapshot** per product at `proc_period1` |
| `active_flag` | cid | `cid` | Was the customer active last month? (drives css decline rule) |

**Why behavioral is on `cid` not `aid`:** payment history is a *customer* signal — a customer with two loans shares one behavioral profile. The model learns "how this person pays," not "how this loan pays."

**Column count > 150** confirms behavioral + aggregates landed. Spot-check `agr3_Mean_CMaxI_Days` and `act_cins_utl` — they should be non-null for customers with ins history.


In [23]:
abt_month.head()

,cid,aid,product,period,act_age,act_cc,act_loaninc,app_income,app_loan_amount,app_n_installments,...,act_ccss_min_lninst,act_ccss_utl,act_ccss_dueutl,act_ccss_cc,act_ccss_seniority,act_ccss_min_seniority,act_ccss_n_loans_hist,act_ccss_n_statC,act_ccss_n_statB,act_cus_active
0,0000000713,css1975020100063,css,197502,79.0,0.846715,12.165450,411.0,5000.0,24.0,...,9.0,0.437500,0.000000,0.846715,49.0,7.0,6.0,NaN,4.0,1.0
1,0000001381,css1975020100098,css,197502,69.0,1.021834,10.917031,458.0,5000.0,24.0,...,6.0,0.388889,0.027778,1.021834,19.0,4.0,3.0,NaN,NaN,1.0
2,0000002567,css1975020100120,css,197502,49.0,0.590922,1.375516,3635.0,5000.0,24.0,...,7.0,0.333333,0.062500,0.590922,28.0,4.0,5.0,1.0,NaN,1.0
3,0000002718,css1975020100123,css,197502,73.0,0.722408,5.574136,897.0,5000.0,24.0,...,17.0,0.291667,0.041667,0.722408,9.0,9.0,1.0,NaN,NaN,1.0
4,0000000089,css1975020200009,css,197502,42.0,0.411944,0.848320,5894.0,5000.0,24.0,...,1.0,0.534722,0.006944,0.411944,60.0,4.0,13.0,6.0,1.0,1.0


In [ ]:
def apply_decision_engine_v1(abt_month: pd.DataFrame,
                              proc_period: str,
                              sim_params: dict) -> pd.DataFrame:
    """Benchmark v1 decision engine — no PD scorecard yet (Phase 5 adds that).

    Actions (per application row):
    1. Default decision='A', decline_reason='999ok'.
    2. If proc_period < burn_in_before → keep approve (burn-in months).
    3. Else if product=='css' and act_cus_active != 1 → decline with reason
       '998 not active customer' (mandatory rule for entire project).
    4. Return slim decision table (aid-level, not full abt).
    """
    decision = pd.Series("A", index=abt_month.index)
    decline_reason = pd.Series("999ok", index=abt_month.index)

    burn_in = proc_period < sim_params["burn_in_before"]
    not_active_css = (
        (abt_month["product"] == "css") & (abt_month["act_cus_active"] != 1)
    )

    decline_mask = not_active_css & (not burn_in)
    decision.loc[decline_mask] = "N"
    decline_reason.loc[decline_mask] = "998 not active customer"

    out = abt_month[["cid", "aid", "product", "period",
                     "app_loan_amount", "app_n_installments"]].copy()
    out["decision"] = decision.values
    out["decline_reason"] = decline_reason.values
    return out


def append_approved_to_pool(approved_tx: pd.DataFrame,
                            transactions: pd.DataFrame,
                            decision_month: pd.DataFrame,
                            proc_period: str) -> pd.DataFrame:
    """Grow the approved transaction pool after each simulation month.

    Actions:
    1. Select loans originated this month: fin_period == proc_period (NOT snapshot period).
    2. Keep only aids with decision == 'A'.
    3. Concatenate their full transaction histories into approved_tx.

    This is the reject-inference mechanism: rejected applicants' future payment
    history never enters the pool that feeds next month's behavioral features.
    """
    month_trans = transactions[transactions["fin_period"] == proc_period]
    approved_aids = decision_month.loc[decision_month["decision"] == "A", "aid"]
    month_trans_approved = month_trans[month_trans["aid"].isin(approved_aids)]
    return pd.concat([approved_tx, month_trans_approved], ignore_index=True)


### Decision engine v1 & approved pool — creating reject inference

**`apply_decision_engine_v1`** — benchmark strategy before any PD scorecard exists:

| Rule | Effect |
|------|--------|
| `proc_period < burn_in_before` | Approve everyone — early months need history to accumulate |
| `css` + `act_cus_active ≠ 1` | Decline with `998 not active customer` — mandatory for entire project |
| Everything else | Approve |

Ins applications are always approved in v1. Css requires an active customer relationship.

**`append_approved_to_pool`** — the feedback loop that makes simulation *stateful*:

1. Take loans **originated** this month (`fin_period == proc_period`).
2. Keep only aids with `decision == 'A'`.
3. Append their full transaction history to `approved_tx`.

**Why this matters for modeling:** next month's behavioral features only see payment history from **accepted** loans. Rejected applicants' future behavior is invisible → the training sample in Phase 4 (`decision='A'`) is biased. That's **reject inference** — a standard interview topic.

**`_resolve_seed_period`:** before the loop starts, seed the pool with all loans from the earliest `fin_period` (`197001`) so month 1 applicants already have 5 years of burn-in history.


In [25]:
def _resolve_seed_period(transactions: pd.DataFrame, sim_params: dict) -> str:
    """Return seed_period from sim_params, or earliest fin_period if None."""
    seed = sim_params.get("seed_period")
    if seed is None:
        return sorted(transactions["fin_period"].unique())[0]
    return seed


def run_simulation(production: pd.DataFrame,
                   transactions: pd.DataFrame,
                   default_df: pd.DataFrame,
                   params: dict,
                   sim_params: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Month-by-month ABT assembly with approved-pool feedback loop.

    Actions:
    1. Seed approved_tx from loans originated at seed_period (burn-in history).
    2. For each proc_period in [start_period, end_period]:
       a. Build behavioral features from approved_tx (not raw transactions).
       b. Build customer aggregates (product, cross-product, active flag).
       c. Assemble abt_month (one row per application).
       d. Apply decision engine v1.
       e. Append approved loans to pool for next month.
    3. Concatenate all months, attach default3/6/12 labels → abt_app.
    """
    seed_period = _resolve_seed_period(transactions, sim_params)
    approved_tx = transactions[transactions["fin_period"] == seed_period].copy()

    all_periods = sorted(production["period"].unique())
    target_periods = [
        p for p in all_periods
        if sim_params["start_period"] <= p <= sim_params["end_period"]
    ]

    all_abt = []
    all_decisions = []

    for proc_period in target_periods:
        proc_period1 = derive_proc_period1(proc_period)
        month_prod = production[production["period"] == proc_period]
        cust_unique = get_cust_unique(production, proc_period)

        behavioral = build_behavioral_for_month(
            approved_tx, production, proc_period, proc_period1, params
        )
        agg_all = build_cust_all_agg(approved_tx, month_prod, cust_unique, proc_period)
        agg_ins = build_cust_product_agg(approved_tx, cust_unique, proc_period1, "ins")
        agg_css = build_cust_product_agg(approved_tx, cust_unique, proc_period1, "css")
        active_flag = build_cust_active_flag(approved_tx, proc_period1)

        abt_month = assemble_abt_month(
            month_prod, behavioral, agg_all, agg_ins, agg_css, active_flag, proc_period
        )
        decisions_month = apply_decision_engine_v1(abt_month, proc_period, sim_params)

        approved_tx = append_approved_to_pool(
            approved_tx, transactions, decisions_month, proc_period
        )

        all_abt.append(abt_month)
        all_decisions.append(decisions_month)

        n_apps = len(abt_month)
        n_appr = (decisions_month["decision"] == "A").sum()
        print(
            f"{proc_period} -> {proc_period1}: {n_apps:>5} apps, "
            f"{n_appr:>5} approved, pool={len(approved_tx):>7,}"
        )

    abt = pd.concat(all_abt, ignore_index=True)
    decisions = pd.concat(all_decisions, ignore_index=True)

    abt_app = abt.merge(
        default_df[["aid", "default3", "default6", "default12"]],
        on="aid",
        how="left",
    )
    return abt_app, decisions


In [26]:
# Dev slice — 3 months before full run
SIM_PARAMS_DEV = {**SIM_PARAMS, "start_period": "197502", "end_period": "197504"}

abt_app_dev, decisions_dev = run_simulation(
    production, transactions, default_df, PARAMS, SIM_PARAMS_DEV
)

print("Dev abt_app:", abt_app_dev.shape)
print("Dev decisions:", decisions_dev.shape)
print("Dev periods:", sorted(abt_app_dev["period"].unique()))
assert len(abt_app_dev["period"].unique()) == 3, "dev run should cover 3 months"
assert abt_app_dev["aid"].is_unique
assert (decisions_dev["decision"] == "N").any(), "expect some css inactive declines"


/var/folders/w7/g4w1_48n7hq28j0995d9c_cr0000gn/T/ipykernel_71700/2722653946.py:21: DeprecationWarning: Bitwise inversion '~' on bool is deprecated. This returns the bitwise inversion of the underlying int object and is usually not what you expect from negating a bool. Use the 'not' operator for boolean negation or ~int(x) if you really want the bitwise inversion of the underlying int.
  decline_mask = not_active_css & (~burn_in)
/var/folders/w7/g4w1_48n7hq28j0995d9c_cr0000gn/T/ipykernel_71700/2722653946.py:21: DeprecationWarning: Bitwise inversion '~' on bool is deprecated. This returns the bitwise inversion of the underlying int object and is usually not what you expect from negating a bool. Use the 'not' operator for boolean negation or ~int(x) if you really want the bitwise inversion of the underlying int.
  decline_mask = not_active_css & (~burn_in)


197502 -> 197501:   301 apps,   142 approved, pool=  5,578
197503 -> 197502:   315 apps,   156 approved, pool=  8,261
197504 -> 197503:   308 apps,   146 approved, pool= 10,905
Dev abt_app: (924, 208)
Dev decisions: (924, 8)
Dev periods: ['197502', '197503', '197504']


/var/folders/w7/g4w1_48n7hq28j0995d9c_cr0000gn/T/ipykernel_71700/2722653946.py:21: DeprecationWarning: Bitwise inversion '~' on bool is deprecated. This returns the bitwise inversion of the underlying int object and is usually not what you expect from negating a bool. Use the 'not' operator for boolean negation or ~int(x) if you really want the bitwise inversion of the underlying int.
  decline_mask = not_active_css & (~burn_in)


### What the 3-month dev slice tells us

The dev run (`197502–197504`) is a **fast parity gate** before the ~2 min full loop.

- **3 distinct `period` values** → the loop bug (early `return`) is fixed; all months accumulate.
- **Some `decision='N'`** → the css inactive rule fires; if every row were `A`, check `act_cus_active` join.
- **Pool size grows each month** → `append_approved_to_pool` is feeding history forward (reject-inference mechanism).

If dev passes, run the full cell below. If not, fix here — don't port broken logic to Kedro.


In [27]:
abt_app_dev

,cid,aid,product,period,act_age,act_cc,act_loaninc,app_income,app_loan_amount,app_n_installments,...,act_ccss_cc,act_ccss_seniority,act_ccss_min_seniority,act_ccss_n_loans_hist,act_ccss_n_statC,act_ccss_n_statB,act_cus_active,default3,default6,default12
0,0000000713,css1975020100063,css,197502,79.0,0.846715,12.165450,411.0,5000.0,24.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0000001381,css1975020100098,css,197502,69.0,1.021834,10.917031,458.0,5000.0,24.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0
2,0000002567,css1975020100120,css,197502,49.0,0.590922,1.375516,3635.0,5000.0,24.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0
3,0000002718,css1975020100123,css,197502,73.0,0.722408,5.574136,897.0,5000.0,24.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
4,0000000089,css1975020200009,css,197502,42.0,0.411944,0.848320,5894.0,5000.0,24.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
919,0000000340,ins1975043000012,ins,197504,47.0,0.202078,0.318166,5582.0,1776.0,12.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
920,0000006865,ins1975043000021,ins,197504,48.0,0.310051,7.441227,1761.0,13104.0,24.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
921,0000000282,ins1975043000023,ins,197504,56.0,0.489412,2.202353,850.0,1872.0,12.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
922,0000006866,ins1975043000031,ins,197504,54.0,0.527363,1.684909,1809.0,3048.0,12.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0


In [ ]:
# Full simulation (197501–198712) — expect ~2–3 min
abt_app, decisions = run_simulation(production, transactions, default_df, PARAMS, SIM_PARAMS)

# --- Parity checks (plan.md Phase 3) ---
print(f"abt_app shape:     {abt_app.shape}")
print(f"Period range:      {abt_app['period'].min()} → {abt_app['period'].max()}")
print(f"Unique aids:         {abt_app['aid'].is_unique}")
print(f"Decisions shape:   {decisions.shape}")
print(f"Declined (N):        {(decisions['decision'] == 'N').sum():,}")

# default12: drop indeterminate (.i / .d) before computing bad rate
default12_num = (
    abt_app["default12"]
    .replace({".i": np.nan, ".d": np.nan})
    .astype(float)
)
print(f"default12 bad rate: {default12_num.mean():.4f}  (expect ~0.41 on this window; plan ~0.49 is full-population slide figure)")

assert abt_app["aid"].is_unique
assert abt_app["period"].min() == "197501"
assert abt_app["period"].max() == "198712"
assert (decisions["decision"] == "N").any()

# Save for Kedro parity / Phase 4
abt_app.to_parquet("../data/04_feature/abt_app.parquet", index=False)
decisions.to_parquet("../data/04_feature/decisions.parquet", index=False)
print("Saved abt_app.parquet and decisions.parquet")


### What the full run tells us

| Metric | Your run | How to read it |
|--------|----------|----------------|
| **Rows** | ~49,224 | One row per **application** in `197501–198712` — matches production volume in the modeling window |
| **Columns** | ~208 | `production` (20) + behavioral (157) + customer aggregates (~30) — fewer than reference 220 because we have not yet added cross-sell response fields |
| **default12 ≈ 0.41** | After dropping `.i`/`.d` | Bad rate on **all** simulated applications; the plan's ~49% figure is the full licensed population including pre-window loans |
| **~7k declines** | `decision='N'` | Css applicants whose customer was not active — the mandatory `998` rule |

**Interview line:** the simulated population bad rate is high because we approve almost everyone in v1. Strategy 1 (Phase 5) tightens that and creates the accepted-only training sample — that's reject inference.

**Next:** port to Kedro (`documents/kedro_phase2_3_port.md`), then Phase 4 bins these ~200 features into WOE and trains PD Ins / PD Css separately.
